## Bootstrap (click Run All — no setup required)

This cell makes the notebook self-installing. It:

1. Finds the Lunar-V2 repo root and puts it on `sys.path`.
2. Installs any missing third-party packages (`numpy`, `scipy`, `numba`, `matplotlib`, plus extras) into the current kernel.
3. Downloads any external data files this notebook needs.

You can re-run it any time; it's a no-op if everything is already present.

In [ ]:
# === Lunar-V2 notebook bootstrap — safe to re-run ===
import sys, pathlib
# Locate the repo root even if this notebook is opened from a weird CWD.
_here = pathlib.Path.cwd().resolve()
for _p in (_here, *_here.parents):
    if (_p / 'pyproject.toml').is_file() and (_p / 'lunar' / '_bootstrap.py').is_file():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break
else:
    raise RuntimeError('Could not find Lunar-V2 repo root from ' + str(_here))

from lunar import _bootstrap as boot
boot.ensure_lunar(extra=('spiceypy',))

boot.ensure_spice_kernels()


# SPICE-Driven Insolation Pipeline

Demonstrates the full pipeline from NAIF SPICE ephemeris → insolation series
→ 1-D thermal solver at a real lunar surface point.

**Requires**:
- `data/spice/` populated with the 6 kernel files (see `data/README.md`)
- `spiceypy` installed (`pip install spiceypy`)

If the kernels are missing, cells 2–4 will raise `FileNotFoundError`. The
notebook gracefully skips those sections and falls back to the sinusoidal
proxy used in notebook 02.

**Reference point**: 26.13 °N, 3.63 °E (Apollo 15 site).  
**Time window**: 2024-01-01 → 2024-01-28 (one lunation, 4-hour cadence).

In [ ]:
from __future__ import annotations
import sys, pathlib, importlib

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta

from lunar.grid import make_geometric_grid
from lunar.properties import conductivity_martinez, density_hayne, specific_heat
from lunar.constants import Q_B_EQUATORIAL, EMISSIVITY_DEFAULT
from lunar.solver import PixelInputs, solve_pixel

_HAVE_SPICE = importlib.util.find_spec('spiceypy') is not None
_HAVE_KERNELS = all(
    (pathlib.Path('../data/spice') / k).is_file()
    for k in ('naif0012.tls', 'pck00011.tpc', 'moon_pa_de440_200625.bpc',
              'moon_de440_250416.tf', 'moon_assoc_me.tf', 'de440s.bsp')
)

print(f'spiceypy available  : {_HAVE_SPICE}')
print(f'SPICE kernels found : {_HAVE_KERNELS}')
USE_SPICE = _HAVE_SPICE and _HAVE_KERNELS
print(f'→ Using {"SPICE ephemeris" if USE_SPICE else "sinusoidal proxy"}')

## 1  Build the insolation time series

In [ ]:
LAT_DEG = 26.13   # Apollo 15
LON_DEG =  3.63
S0      = 1361.0  # W m^-2

if USE_SPICE:
    from lunar import ephem

    # One lunation at 4-hour cadence: 2024-01-01 to 2024-01-28
    times_iso = [
        (datetime(2024, 1, 1) + timedelta(hours=4*i)).strftime('%Y-%m-%dT%H:%M:%S')
        for i in range(7 * 24 // 4 * 4)   # 28 days × 6 samples/day = 168 samples
    ]
    et      = ephem.et_from_iso(times_iso)
    elev, az = ephem.solar_elevation_azimuth(et, lat_deg=LAT_DEG, lon_deg=LON_DEG)
    S       = ephem.insolation_series(elev, solar_constant=S0)
    t_s     = (et - et[0]).astype(float)   # seconds from start
    ephem.unload_kernels()

    print(f'SPICE: {len(times_iso)} time steps,  dt = 4 h')
    print(f'  Peak insolation : {S.max():.1f} W m^-2')
    print(f'  Min  elevation  : {np.rad2deg(elev.min()):.1f} deg')
    print(f'  Max  elevation  : {np.rad2deg(elev.max()):.1f} deg')

else:
    # Sinusoidal proxy (identical to notebook 02 but with cos(lat) scaling)
    T_LUNAR = 27.321661 * 86400.0
    N_t     = int(T_LUNAR / 3600.0) + 1
    t_s     = np.linspace(0.0, T_LUNAR, N_t)
    phase   = 2.0 * np.pi * t_s / T_LUNAR
    S       = S0 * np.cos(np.deg2rad(LAT_DEG)) * np.maximum(0.0, np.cos(phase))
    elev    = None
    print('Sinusoidal proxy: SPICE not available')

# Plot insolation
fig, axes = plt.subplots(1, 2 if USE_SPICE else 1, figsize=(13 if USE_SPICE else 8, 4))
if not USE_SPICE:
    axes = [axes]

ax = axes[0]
ax.plot(t_s / 86400.0, S, color='orange', lw=1.5)
ax.set_xlabel('Days from start')
ax.set_ylabel('Insolation [W m⁻²]')
ax.set_title(f'Insolation at {LAT_DEG:.2f}°N, {LON_DEG:.2f}°E')
ax.grid(True, alpha=0.3)

if USE_SPICE and len(axes) > 1:
    ax2 = axes[1]
    ax2.plot(t_s / 86400.0, np.rad2deg(elev), color='steelblue', lw=1.5)
    ax2.axhline(0, color='k', lw=0.8, ls='--')
    ax2.set_xlabel('Days from start')
    ax2.set_ylabel('Solar elevation [deg]')
    ax2.set_title('Solar elevation (SPICE/DE440)')
    ax2.grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig('spice_insolation.png', dpi=150)
print('Saved spice_insolation.png')
plt.close(fig)

## 2  Thermal solver run

In [ ]:
grid = make_geometric_grid(z_max=10.0, dz0=0.003, growth=1.08)

inputs = PixelInputs(
    grid=grid,
    t=t_s,
    bc_mode='radiative',
    insolation=S,
    albedo=0.12,
    emissivity=EMISSIVITY_DEFAULT,
    Q_b=Q_B_EQUATORIAL,
    K_func=lambda T, z: conductivity_martinez(T, z),
    rho_func=lambda z: density_hayne(z),
    cp_func=lambda T: specific_heat(T, model='biele'),
    n_lunations_spinup=10,
    spinup_tol_K=0.01,
)

print(f'Grid: {grid.n_layers} layers,  z_max = {grid.z_mid[-1]:.2f} m')
print('Running spin-up...')
out = solve_pixel(inputs)
print(f'Spin-up: {out.n_spinup_cycles} cycles,  converged={out.converged}')

T_surface = out.T[0, :]
print(f'Surface T: peak={T_surface.max():.1f} K,  min={T_surface.min():.1f} K')

## 3  Compare insolation-driven surface temperature to sinusoidal proxy

In [ ]:
if USE_SPICE:
    # Run the proxy too for direct comparison
    T_LUNAR = 27.321661 * 86400.0
    N_proxy = int(T_LUNAR / 3600.0) + 1
    t_proxy = np.linspace(0.0, T_LUNAR, N_proxy)
    phase   = 2.0 * np.pi * t_proxy / T_LUNAR
    S_proxy = S0 * np.cos(np.deg2rad(LAT_DEG)) * np.maximum(0.0, np.cos(phase))

    inp_proxy = PixelInputs(
        grid=grid, t=t_proxy, bc_mode='radiative',
        insolation=S_proxy, albedo=0.12, emissivity=EMISSIVITY_DEFAULT,
        Q_b=Q_B_EQUATORIAL,
        K_func=lambda T, z: conductivity_martinez(T, z),
        rho_func=lambda z: density_hayne(z),
        cp_func=lambda T: specific_heat(T, model='biele'),
        n_lunations_spinup=10, spinup_tol_K=0.01,
    )
    out_proxy = solve_pixel(inp_proxy)
    T_surface_proxy = out_proxy.T[0, :]

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(t_s / 86400.0, T_surface, lw=1.5, color='firebrick', label='SPICE-driven')
    ax.plot(t_proxy / 86400.0, T_surface_proxy, lw=1.5, color='steelblue',
            ls='--', label='Sinusoidal proxy')
    ax.set_xlabel('Days')
    ax.set_ylabel('Surface temperature [K]')
    ax.set_title(f'Surface T at {LAT_DEG:.2f}°N  —  SPICE vs proxy')
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig('spice_vs_proxy.png', dpi=150)
    print('Saved spice_vs_proxy.png')
    plt.close(fig)
else:
    print('SPICE not available — skipping comparison plot')

## 4  Subsurface temperature snapshots

In [ ]:
fig, ax = plt.subplots(figsize=(7, 8))

i_peak  = int(np.argmax(S))
i_min   = int(np.argmin(S[S > 0]) if (S > 0).any() else np.argmin(S))
i_night = int(np.argmax(S == 0.0) + S[S == 0.0].size // 2) if (S == 0.0).any() else i_min

colors = ['orange', 'royalblue', 'purple']
labels = ['Solar noon', 'Twilight', 'Midnight']

for idx, color, label in zip([i_peak, i_min, i_night], colors, labels):
    ax.plot(out.T[:, idx], out.z * 100.0, color=color, lw=2, label=label)

# Mark Apollo 15 sensor depths
for d in [35, 45, 73, 84, 91, 101, 139]:
    ax.axhline(d, color='gray', lw=0.5, ls=':', alpha=0.6)
    ax.text(50, d + 1.5, f'{d} cm', fontsize=7, color='gray')

ax.invert_yaxis()
ax.set_xlabel('Temperature [K]', fontsize=12)
ax.set_ylabel('Depth [cm]', fontsize=12)
ax.set_title(f'Subsurface T profile — {LAT_DEG:.2f}°N', fontsize=13)
ax.set_ylim(200, 0)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('spice_subsurface_profile.png', dpi=150)
print('Saved spice_subsurface_profile.png')
plt.close(fig)

## 5  Next steps

To scale this to a polar DEM pixel:

```python
from lunar.illumination import load_lola_dem, compute_horizon, is_illuminated

dem     = load_lola_dem('data/dem/LDEM_80S_80MPP_ADJ.TIF', subsample=4)
horizon = compute_horizon(dem, n_azimuth=360)

# For pixel (i, j), build shadow-corrected insolation:
for k, (solar_e, solar_a) in enumerate(zip(elev, az)):
    if is_illuminated(solar_e, solar_a, horizon[i, j], az_centers):
        S_shadowed[k] = insolation_series(np.array([solar_e]))[0]
    else:
        S_shadowed[k] = 0.0
```

Then feed `S_shadowed` into `PixelInputs.insolation` and run `solve_pixel` as above.
The full south-polar pipeline is in `lunar/pipeline.py`.